# Weighted-Mean Vertical Profiles and Initial Parcel Parameters for Skew-T Plotting

Purpose: Aggregate per-year outputs from pressure_level_profiles.py, screen_level_surface_
conditions.py, and initial_parcel_parameters.py into weighted-mean summaries, ready for
Skew-T/log-P plotting (done in a separate notebook).
- Vertical profile step: for each lightning/OT category, combine per-year pressure-level
  temperature/dew point files into a single count-weighted mean profile across all years,
  substituting the lowest pressure level with the count-weighted screen-level value where
  appropriate (e.g. over ocean, where 1000 hPa is below the surface).
- Initial parameter step: combine per-year lifted-parcel-level temperature, dew point,
  and pressure into a count-weighted "total" (all-years) summary row per category,
  appended to the per-year values.

Input:
- Per-year pressure-level profile CSVs (from pressure_level_profiles.py):
  warm_Season_mucape_0_pressure_level_param/1_1_MUCAPE_0_*_<year>.csv
- Per-category screen-level CSVs (from screen_level_surface_conditions.py):
  warm_Season_mucape_0_pressure_level_param/1_screen_level_mean_*.csv
- Combined initial-parameter CSV (from initial_parcel_parameters.py):
  warm_season_initial_parameters/1_initial_param.csv
  

Processing:
1. For each category (6 land/ocean x LS/OT combinations), sum valid-data counts across
   years, then compute a count-weighted mean temperature and dew point at each pressure
   level across all years.
2. Substitute the 1000 hPa level with the count-weighted screen-level
   temperature/dew point/pressure where the average screen pressure is below 1000 hPa.
3. For initial parcel parameters, pivot per-year values by variable, drop duplicate
   (year, variable) rows, and compute a count-weighted "total" row across all years,
   appended as an extra row.

Output:
- Weighted-mean vertical profile CSVs (per category):
  BARRA_data_skew_T/mean_data/MUCAPE_0/with_screen_level_MUCAPE_0_mean_data_<folder>.csv
- Weighted-average initial parameter CSVs (per group/contrast):
  warm_season_initial_parameters/with_weighted_avg_<group>_<contrast>.csv
  (Note: filename for "OT + LS" required a manual correction from "ot_ls" to "ls_ot"
  after generation.)

In [1]:
import pandas as pd
import numpy as np
import os
import re

In [2]:
#this is calculate weighted mean data parameter for cape plot with screen data
import pandas as pd
import glob
import os

years = list(range(2016, 2025))
folders = ['ls_only_land','ls_only_ocean','ls_ot_land','ls_ot_ocean','ot_only_land', 'ot_only_ocean'] #,'ls_ot_ocean'
barra_dir = '/g/data/if69/tp4064/Project_1/categorical_position/BARRA_data_skew_T/warm_Season_mucape_0_pressure_level_param'
# barra_dir_screen = '/g/data/if69/tp4064/Project_1/categorical_position/BARRA_data_skew_T/warm_season_screen_parameters'


for folder in folders:
    print(f"\n========== Processing folder: {folder} ==========")

    # --- Step 1: Sum 'count temp' across all years ---
    file_list = glob.glob(f"{barra_dir}/1_MUCAPE_*_{folder}_*.csv")
    count_df = None
    # print(file_list[0])

    for f in file_list:
        
        df = pd.read_csv(f, index_col=0)
        # print(df)
        col = pd.to_numeric(df['count_temp'], errors='coerce')
        if count_df is None:
            count_df = col.to_frame()
        else:
            count_df['count_temp'] += col
    
        
    # --- Step 2: Compute weighted mean ---
    temp = pd.DataFrame()
    dew  = pd.DataFrame()
    mean_data = pd.DataFrame()

    for year in years:
        
        file = glob.glob(f'{barra_dir}/1_1_MUCAPE_*_{folder}_*{year}.csv')[0]
        
        data = pd.read_csv(file, usecols = ['pressure', 'temperature','dew_point','count_temp','count_dew'])
        # print(data)
        data.set_index(data.columns[0], inplace=True)
         # concating all year files 
        temp[year] = data['temperature'] * data['count_temp'] / count_df['count_temp']
        dew[year]  = data['dew_point']   * data['count_dew']  / count_df['count_temp'] #count_temp == count_dew

    # suming all those years along a pressure rows
        
    mean_data[f'{folder}_temperature'] = temp.sum(axis=1) - 273.15
    mean_data[f'{folder}_dew']         = dew.sum(axis=1)  - 273.15
    
    print(mean_data.reset_index(inplace = True))

    screen_file = glob.glob(f'{barra_dir}/1_screen_level_*_{folder}_*.csv')[0]

    screen_data = pd.read_csv(screen_file)

    avg_temp = ((screen_data['surface_temp']*screen_data['count_temp'])/screen_data['count_temp'].sum()).sum()
    avg_dew = ((screen_data['surface_dew']*screen_data['count_dew'])/screen_data['count_dew'].sum()).sum()
    avg_press = ((screen_data['surface_press']*screen_data['Count_press'])/screen_data['Count_press'].sum()).sum()

    # screen_row = screen_data[screen_data['year'] == 'total']
    # print(screen_row)

    if  (avg_press < 1000):

        #note : for oceans screen level is higher than 1000 hpa which has been not included in it.
        
        mean_data.loc[mean_data['pressure'] == 1000,f'{folder}_temperature'] = (avg_temp -273.15)
        mean_data.loc[mean_data['pressure'] == 1000,f'{folder}_dew'] = (avg_dew-273.15)
        mean_data.loc[mean_data['pressure'] == 1000,'pressure'] = avg_press

    print(mean_data)
    
    mean_data.to_csv(f"/g/data/if69/tp4064/Project_1/categorical_position/BARRA_data_skew_T/mean_data/MUCAPE_0/with_screen_level_MUCAPE_0_mean_data_{folder}.csv")
# # print(f"\nSaved → {barra_dir}/mean_data_ot_only.csv")


========== Processing folder: ls_only_land ==========
None
     pressure  ls_only_land_temperature  ls_only_land_dew
0  985.090342                 31.172459         20.788519
1  950.000000                 28.876347         19.097280
2  925.000000                 27.051263         18.166294
3  850.000000                 21.362760         15.310997
4  700.000000                 10.164207          6.952283
5  600.000000                  2.525256         -1.364199
6  500.000000                 -4.849885        -13.606244
7  400.000000                -14.947304        -27.448335
8  300.000000                -29.422646        -41.595436
9  200.000000                -51.292265        -60.469840

========== Processing folder: ls_only_ocean ==========
None
   pressure  ls_only_ocean_temperature  ls_only_ocean_dew
0    1000.0                  28.533020          24.168181
1     950.0                  24.721809          22.232602
2     925.0                  23.483946          20.532728
3     850

In [9]:
# to modify initial parameters
from pathlib import Path

barra_initial_dir = Path('/g/data/if69/tp4064/Project_1/categorical_position/BARRA_data_skew_T/warm_season_initial_parameters')

data_initial = pd.read_csv(barra_initial_dir/"1_initial_param.csv")

for i, g in data_initial.groupby([data_initial['group'],data_initial['contrast']] ):

    print(i)
    # print(g)

    #some how there is duplicate in OT + LS group and OT only group, which needs to be fixed instead of rereunning codes.
    g = g.drop_duplicates(subset=['year', 'Variable'])
    # dupes = g[g.duplicated(subset=['year', 'Variable'], keep=False)]
    # print(dupes.sort_values(['year', 'Variable']))


    pivoted = g.pivot(index = 'year', columns = 'Variable', values = ['mean', 'nonnan_count'])

    pivoted.columns= [f'{col[1]}_{col[0]}' if col[1] else col[0] for col in pivoted.columns]

    pivoted = pivoted.reset_index()

    print(i)

    avg_temp = ((pivoted['MULPLtemp_mean']*pivoted['MULPLtemp_nonnan_count'])/pivoted['MULPLtemp_nonnan_count'].sum()).sum()
    avg_dew = ((pivoted['MULPLdew_mean']*pivoted['MULPLdew_nonnan_count'])/pivoted['MULPLdew_nonnan_count'].sum()).sum()
    avg_press = ((pivoted['MULPLpres_mean']*pivoted['MULPLpres_nonnan_count'])/pivoted['MULPLpres_nonnan_count'].sum()).sum()

    summary_row = pd.DataFrame([{
        'year':'total',
        'MULPLtemp_mean':avg_temp-273.15, # in deg celsius
        'MULPLdew_mean':avg_dew-273.15, # in deg celsius
        'MULPLpres_mean':avg_press/100, # in hpa
        'MULPLtemp_nonnan_count':pivoted['MULPLtemp_nonnan_count'].sum(),
        'MULPLdew_nonnan_count':pivoted['MULPLdew_nonnan_count'].sum(),
        'MULPLpres_nonnan_count':pivoted['MULPLpres_nonnan_count'].sum(),
    }])

    print(summary_row)

    result = pd.concat([pivoted, summary_row], ignore_index=True)
    # file_name = f'{i[0]}_{i[1]}'.lower().replace(" ", "_").replace(" + ", "_") + ".csv"

    file_name = re.sub(r'[^a-z0-9]+', '_', f'{i[0]}_{i[1]}'.lower()).strip('_') + ".csv"

    # for OT + LS, i did manual changes to ot_ls to ls_ot

    
    result.to_csv(barra_initial_dir/str("with_weighted_avg_"+file_name), index = False)



('LS only', 'Land')
('LS only', 'Land')
    year  MULPLtemp_mean  MULPLdew_mean  MULPLpres_mean  \
0  total       30.859975      20.647537      978.417024   

   MULPLtemp_nonnan_count  MULPLdew_nonnan_count  MULPLpres_nonnan_count  
0               5406500.0              5406500.0               5406500.0  
('LS only', 'Ocean')
('LS only', 'Ocean')
    year  MULPLtemp_mean  MULPLdew_mean  MULPLpres_mean  \
0  total       28.988004      24.410902     1004.856459   

   MULPLtemp_nonnan_count  MULPLdew_nonnan_count  MULPLpres_nonnan_count  
0               2131999.0              2131999.0               2131999.0  
('OT + LS', 'Land')
('OT + LS', 'Land')
    year  MULPLtemp_mean  MULPLdew_mean  MULPLpres_mean  \
0  total        29.48025      23.351916      983.510763   

   MULPLtemp_nonnan_count  MULPLdew_nonnan_count  MULPLpres_nonnan_count  
0                509297.0               509297.0                509297.0  
('OT + LS', 'Ocean')
('OT + LS', 'Ocean')
    year  MULPLtemp_mean  MUL